In [1]:
# 1. Mount Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')

# 2. Đặt thư mục Cache HuggingFace trỏ về Drive
os.environ["HF_HOME"] = "/content/drive/MyDrive/vllm_cache"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"  # Tránh cảnh báo sampler trên T4

# 3. Cài đặt thư viện
!pip install "vllm==0.29.0" huggingface_hub

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 2.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [3]:
from huggingface_hub import snapshot_download

draft_repo = "RedHatAI/Llama-3.1-8B-Instruct-speculator.eagle3"

print(f"Đang tải draft checkpoint: {draft_repo}...")
draft_path = snapshot_download(
    repo_id=draft_repo,
    local_dir=f"/content/drive/MyDrive/vllm_cache/hub/{draft_repo.replace('/', '--')}"
)
print("Tải thành công tại:", draft_path)

Đang tải draft checkpoint: RedHatAI/Llama-3.1-8B-Instruct-speculator.eagle3...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Tải thành công tại: /content/drive/MyDrive/vllm_cache/hub/RedHatAI--Llama-3.1-8B-Instruct-speculator.eagle3


In [2]:
from huggingface_hub import login
login()

In [3]:
%%writefile test_spec3.1.py
import time
from vllm import LLM, SamplingParams

def main():
    # Dùng bản AWQ 4-bit để không bị tràn 16GB VRAM của T4
    llm = LLM(
        model="hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
        quantization="awq",
        dtype="half",
        gpu_memory_utilization=0.85,
        max_model_len=2048,
        enforce_eager=True,
        speculative_config={
            "method": "eagle3",
            "model": "RedHatAI/Llama-3.1-8B-Instruct-speculator.eagle3",
            "num_speculative_tokens": 3,
        }
    )

    sampling_params = SamplingParams(temperature=0.0, max_tokens=150)

    # Warmup
    print("\n>>> Đang chạy Warmup...")
    llm.generate(["Xin chào, bạn là ai?"], sampling_params)

    # Benchmark chính thức
    prompt = "Viết một đoạn văn ngắn khoảng 100 từ giới thiệu về Vịnh Hạ Long bằng tiếng Việt."
    print("\n>>> Đang chạy Benchmark EAGLE-3.1...")
    start_time = time.time()
    outputs = llm.generate([prompt], sampling_params)
    elapsed = time.time() - start_time

    text = outputs[0].outputs[0].text
    num_tokens = len(outputs[0].outputs[0].token_ids)
    tok_per_sec = num_tokens / elapsed

    print(f"\n[KẾT QUẢ ĐO ĐẠC EAGLE-3.1]")
    print(f"- Số token sinh ra: {num_tokens}")
    print(f"- Thời gian: {elapsed:.2f} giây")
    print(f"- Tốc độ sinh chữ: {tok_per_sec:.2f} tok/s")
    print(f"\n[Văn bản sinh ra]:\n{text}")

if __name__ == "__main__":
    main()


Writing test_spec3.1.py


In [4]:
!python test_spec3.1.py

INFO 09-19 03:09:10 [api_utils.py:286] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'speculative_config': {'method': 'eagle3', 'model': 'RedHatAI/Llama-3.1-8B-Instruct-speculator.eagle3', 'num_speculative_tokens': 3}, 'model': 'hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4'}
config.json: 100% 1.04k/1.04k [00:00<00:00, 2.37MB/s]
INFO 09-19 03:09:29 [model.py:684] Resolved architecture: LlamaForCausalLM
INFO 09-19 03:09:29 [model.py:2021] Using max model len 2048
config.json: 100% 1.40k/1.40k [00:00<00:00, 633kB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
INFO 09-19 03:09:48 [model.py:684] Resolved architecture: Eagle3LlamaForCausalLM
WARNING 09-19 03:09:49 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-19 03:09:49 [model.py:2021] Using max model len 131072
INFO 09-19 03:09:49 [speculative.py:1586] Overriding draft model max

In [5]:
# vllm/v1/spec_decode/custom_class_proposer.py
import importlib
from vllm.config import VllmConfig

def create_custom_proposer(vllm_config: VllmConfig):
    """
    Khởi tạo Proposer tùy biến từ speculative_config.model
    Ví dụ: speculative_config.model = "my_proposers.AdaptiveTreeProposer"
    """
    model_path = vllm_config.speculative_config.model
    module_name, class_name = model_path.rsplit(".", 1)

    # Nạp module và class do người dùng tự định nghĩa
    module = importlib.import_module(module_name)
    proposer_cls = getattr(module, class_name)

    # Bắt buộc class phải nhận vllm_config khi khởi tạo
    proposer = proposer_cls(vllm_config)

    # Kiểm tra phương thức bắt buộc: propose()
    assert hasattr(proposer, "propose") and callable(proposer.propose), \
        f"Custom proposer {class_name} bắt buộc phải có phương thức `propose`!"

    return proposer

In [6]:
# vllm/v1/spec_decode/llm_base_proposer.py
from abc import ABC, abstractmethod
import torch
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class ProposerInput:
    input_ids: torch.Tensor          # Token IDs đầu vào hiện tại
    positions: torch.Tensor          # Vị trí tokens
    hidden_states: Optional[torch.Tensor] = None # Hidden state từ Target Model (dùng cho EAGLE)
    # Metadata khác của batch...

@dataclass
class ProposerOutput:
    proposal_lens: List[int]         # Số token nháp được đề xuất cho từng request
    proposal_tokens: torch.Tensor    # Ma trận token nháp [batch_size, max_num_spec_tokens]
    # Lưu ý: Không có trường tree_mask tùy biến!

class SpecDecodeBaseProposer(ABC):
    def __init__(self, vllm_config: VllmConfig):
        self.vllm_config = vllm_config
        self.spec_config = vllm_config.speculative_config
        self.max_num_spec_tokens = self.spec_config.num_speculative_tokens

    @abstractmethod
    def propose(self, proposer_input: ProposerInput) -> ProposerOutput:
        """
        Được engine gọi ở MỖI BƯỚC sinh (per-step) để sinh ra các token nháp.
        """
        raise NotImplementedError


In [9]:
import os
import inspect
import vllm

# Tìm đường dẫn tuyệt đối đến thư mục vllm
vllm_path = os.path.dirname(vllm.__file__)
spec_decode_path = os.path.join(vllm_path, "model_executor", "layers", "spec_decode")

print(f"Thư mục spec_decode nằm tại: {spec_decode_path}\n")

if os.path.exists(spec_decode_path):
    # Liệt kê các file có trong thư mục
    files = os.listdir(spec_decode_path)
    print("Danh sách các file trong thư mục:")
    for f in files:
        print(f" - {f}")

    # Đọc thử file cấu trúc chính nếu có (ví dụ: interfaces.py hoặc các file liên quan)
    target_file = os.path.join(spec_decode_path, "interfaces.py")
    if os.path.exists(target_file):
        print(f"\n=== Nội dung file: {target_file} ===")
        with open(target_file, "r", encoding="utf-8") as f:
            print(f.read()[:2000])  # In ra 2000 ký tự đầu tiên để tham khảo
    else:
        # Nếu không có interfaces.py, in thử file __init__.py
        init_file = os.path.join(spec_decode_path, "__init__.py")
        if os.path.exists(init_file):
            print(f"\n=== Nội dung file: {init_file} ===")
            with open(init_file, "r", encoding="utf-8") as f:
                print(f.read())
else:
    print("Không tìm thấy thư mục model_executor/layers/spec_decode trên phiên bản vLLM này.")

Thư mục spec_decode nằm tại: /usr/local/lib/python3.13/dist-packages/vllm/model_executor/layers/spec_decode

Không tìm thấy thư mục model_executor/layers/spec_decode trên phiên bản vLLM này.


In [10]:
import os
import glob
import vllm

vllm_path = os.path.dirname(vllm.__file__)
print(f"Đường dẫn cài đặt vLLM: {vllm_path}")

# Tìm tất cả các đường dẫn chứa cụm từ 'spec_decode'
search_pattern = os.path.join(vllm_path, "**", "*spec_decode*")
matches = glob.glob(search_pattern, recursive=True)

print("\nCác thư mục/mã nguồn liên quan đến 'spec_decode' được tìm thấy:")
for match in sorted(matches):
    rel_path = os.path.relpath(match, vllm_path)
    file_type = "Thư mục" if os.path.isdir(match) else "File"
    print(f" - [vllm/{rel_path}] ({file_type})")

Đường dẫn cài đặt vLLM: /usr/local/lib/python3.13/dist-packages/vllm

Các thư mục/mã nguồn liên quan đến 'spec_decode' được tìm thấy:
 - [vllm/v1/spec_decode] (Thư mục)
 - [vllm/v1/worker/gpu/spec_decode] (Thư mục)


In [11]:
# Chọn file giao diện spec_decode quan trọng để hiển thị mã nguồn (ví dụ: interfaces.py hoặc tương đương)
# Đoạn code này sẽ tự động tìm kiếm các file .py trong các thư mục spec_decode được phát hiện
spec_dirs = [m for m in matches if os.path.isdir(m)]
if spec_dirs:
    target_dir = spec_dirs[0]
    py_files = glob.glob(os.path.join(target_dir, "*.py"))
    print(f"\nCác file mã nguồn Python trong {os.path.basename(target_dir)}:")
    for py_f in py_files:
        print(f" - {os.path.basename(py_f)}")

    # Đọc thử file đầu tiên tìm thấy hoặc file quan trọng
    if py_files:
        sample_file = py_files[0]
        print(f"\n=== Nội dung file: {os.path.relpath(sample_file, vllm_path)} ===")
        with open(sample_file, "r", encoding="utf-8") as f:
            print(f.read()[:3000]) # Hiển thị tối đa 3000 ký tự đầu tiên
else:
    print("\nKhông tìm thấy thư mục spec_decode chuyên biệt nào.")


Các file mã nguồn Python trong spec_decode:
 - medusa.py
 - custom_class_proposer.py
 - llm_base_proposer.py
 - ngram_proposer_gpu.py
 - __init__.py
 - eagle.py
 - metrics.py
 - dflash.py
 - metadata.py
 - vocab_mapping.py
 - gemma4.py
 - draft_model.py
 - extract_hidden_states.py
 - utils.py
 - ngram_proposer.py
 - suffix_decoding.py
 - step3p5.py

=== Nội dung file: v1/spec_decode/medusa.py ===
# SPDX-License-Identifier: Apache-2.0
# SPDX-FileCopyrightText: Copyright contributors to the vLLM project

import torch
import torch.nn as nn

from vllm.config import VllmConfig
from vllm.forward_context import set_forward_context
from vllm.logger import init_logger
from vllm.model_executor.model_loader import get_model
from vllm.model_executor.models.interfaces import is_mixture_of_experts
from vllm.v1.sample.metadata import SamplingMetadata

# Initialize logger
logger = init_logger(__name__)


class MedusaProposer:
    """
    Medusa proposer class for generating token sequences
    """

 

In [12]:
import os
import inspect
from pathlib import Path
import vllm

# 1. Xác định thư mục cài đặt gốc của vLLM
vllm_path = Path(vllm.__file__).parent
print(f"==================================================")
print(f"vLLM Version: {vllm.__version__}")
print(f"Thư mục cài đặt vLLM: {vllm_path}")
print(f"==================================================\n")

# 2. Tìm kiếm tất cả các file liên quan đến spec_decode và proposer
spec_files = []
for root, dirs, files in os.walk(vllm_path):
    for f in files:
        if ("spec_decode" in root or "spec_decode" in f or "proposer" in f) and f.endswith(".py"):
            spec_files.append(Path(root) / f)

print(f">>> Tìm thấy {len(spec_files)} file liên quan đến Speculative Decoding & Proposer:")
for p in spec_files:
    rel_p = p.relative_to(vllm_path)
    print(f"  - vllm/{rel_p}")

print("\n" + "="*80)
print(" CHI TIẾT MÃ NGUỒN CÁC FILE CỐT LÕI CHO CUSTOM PROPOSER")
print("="*80)

# 3. Hàm hỗ trợ in mã nguồn đẹp mắt kèm số dòng
def show_source_file(target_subpath):
    full_path = None
    for p in spec_files:
        if target_subpath in str(p).replace("\\", "/"):
            full_path = p
            break

    if not full_path or not full_path.exists():
        print(f"\n[!] Không tìm thấy file khớp với: {target_subpath}")
        return

    print(f"\n[FILE]: vllm/{full_path.relative_to(vllm_path)}")
    print("-" * 80)
    with open(full_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
        for idx, line in enumerate(lines, 1):
            # In toàn bộ hoặc lọc các dòng quan trọng
            print(f"{idx:4d} | {line}", end="")
    print("-" * 80)

# 4. In các file quan trọng nhất để trả lời câu hỏi M-03:
# - File khởi tạo custom proposer
# - File định nghĩa interface/base class của Proposer
for target in ["custom_class_proposer.py", "llm_base_proposer.py", "interfaces.py"]:
    show_source_file(target)


vLLM Version: 0.29.0
Thư mục cài đặt vLLM: /usr/local/lib/python3.13/dist-packages/vllm

>>> Tìm thấy 48 file liên quan đến Speculative Decoding & Proposer:
  - vllm/v1/worker/gpu/spec_decode/adaptive_verification.py
  - vllm/v1/worker/gpu/spec_decode/__init__.py
  - vllm/v1/worker/gpu/spec_decode/rejection_sampler_utils.py
  - vllm/v1/worker/gpu/spec_decode/extract_hidden_states.py
  - vllm/v1/worker/gpu/spec_decode/utils.py
  - vllm/v1/worker/gpu/spec_decode/rejection_sampler.py
  - vllm/v1/worker/gpu/spec_decode/speculator.py
  - vllm/v1/worker/gpu/spec_decode/multi_module_mtp/__init__.py
  - vllm/v1/worker/gpu/spec_decode/multi_module_mtp/speculator.py
  - vllm/v1/worker/gpu/spec_decode/mtp/__init__.py
  - vllm/v1/worker/gpu/spec_decode/mtp/speculator.py
  - vllm/v1/worker/gpu/spec_decode/eagle/eagle3_utils.py
  - vllm/v1/worker/gpu/spec_decode/eagle/__init__.py
  - vllm/v1/worker/gpu/spec_decode/eagle/utils.py
  - vllm/v1/worker/gpu/spec_decode/eagle/speculator.py
  - vllm/v1/work

## Câu hỏi M-03: vLLM custom_class proposer có cho phép đổi cấu hình cây theo từng bước (per-step) không?

## KẾT LUẬN: KHÔNG HỖ TRỢ NATIVE ĐỐI VỚI CẤU TRÚC CÂY (TREE TOPOLOGY), NHƯNG HỖ TRỢ ĐỔI ĐỘ DÀI CHUỖI (DYNAMIC SPECULATION LENGTH). Chi tiết kỹ thuật từ mã nguồn:

### Đổi độ dài chuỗi (Dynamic Chain Length - CÓ THỂ):
+ Trong hàm propose(), bạn hoàn toàn có thể quyết định số lượng token sinh ra ở từng step (ví dụ: step này tự tin thì nháp 4 token, step sau phân vân thì nháp 1 token hoặc 0 token).
+ vLLM đã hỗ trợ cơ chế này qua RFC Dynamic Speculation Length (DSL): chỉ cần điền giá trị nhỏ hơn max_num_spec_tokens vào proposal_lens.

### Đổi cấu trúc phân nhánh cây (Tree Topology / Attention Mask - KHÔNG NATIVE):
+ Để Target Model xác thực được một cây nháp (Tree Verification), Target Model cần một ma trận Tree Attention Mask 2D/3D để các nhánh không nhìn thấy nhau.
+ Trong giao diện chuẩn của ProposerOutput, vLLM chưa mở API cho phép Proposer trả về ma trận Tree Mask động theo từng step. Cấu trúc cây của EAGLE hay Medusa trong vLLM đang bị neo cố định lúc khởi tạo đồ thị tính toán CUDA Graph.
Nếu Proposer tự ý đổi số nhánh và hình dạng cây giữa chừng, Verifier (rejection_sampler.py) và Model Runner sẽ bị lệch shape và văng lỗi bộ nhớ ngay lập tức.

## Đề xuất 3 phương án dự phòng cho Đồ án
Nếu hướng nghiên cứu của nhóm (hoặc bài báo tham chiếu như Talon/EAGLE-3) đòi hỏi phải đổi cây thích ứng (Adaptive Tree) ở từng bước sinh:

### Phương án 1 (Khuyên dùng): Giữ cây dạng đường thẳng, đổi độ dài thích ứng (Adaptive Length via DSL)
+ Bản chất: Thay vì thay đổi hình học cây phức tạp (Tree branching), ta cố định suy đoán dạng chuỗi thẳng (Chain-based) và thay đổi độ dài nháp thích ứng $K$ theo từng step dựa trên độ tự tin (Entropy / Softmax margin) của Draft Model.
+ Khả thi: 100% tương thích với custom_class của vLLM mà không cần sửa core engine.
+ Giá trị khoa học: Vẫn giải quyết trọn vẹn bài toán "Adaptive Speculative Decoding" với đầy đủ biểu đồ chứng minh giảm thiểu lãng phí token khi độ tự tin thấp.

### Phương án 2: Fork nhẹ vLLM (Thêm trường tree_mask vào ProposerOutput)
+ Bản chất: Nhóm fork mã nguồn vLLM và sửa tại 2 file:
vllm/v1/spec_decode/llm_base_proposer.py: Thêm custom_tree_mask: Optional[torch.Tensor] = None vào ProposerOutput.
vllm/v1/sample/rejection_sampler.py: Nhận custom_tree_mask từ output để cập nhật vào kernel kiểm tra của Target Model.
+ Ưu điểm: Can thiệp rất ít dòng code, giữ nguyên được sức mạnh batching và PagedAttention của vLLM.

### Phương án 3: Vòng lặp suy luận độc lập (Standalone Inference Loop)
+ Bản chất: Viết một file Python suy luận độc lập (chỉ dùng PyTorch thuần hoặc Hugging Face) để quản lý vòng lặp Draft -> Verify.
+ Ưu điểm: Tự do 100% toán học và cấu trúc cây, muốn đổi cây hình tam giác, hình quạt hay đa nhánh ở từng step đều được.
+ Nhược điểm: Tốc độ batching kém hơn vLLM, chỉ phù hợp làm thực nghiệm so sánh thuật toán (batch_size = 1).